# Crack Segmentation — Experimental Model Evaluation

This notebook evaluates trained **UNet**, **SegFormer**, and **YOLO-seg** models on the held-out test split.
It computes:
1. **Area-based Segmentation Metrics**: IoU, Dice, Pixel Accuracy, Precision, Recall, Specificity
2. **Raw Skeleton Topology & Centerline Metrics**: clDice, Centerline Precision / Recall / F1 / IoU (@20px), Orientation MAE
3. **Branch-Removed Centerline Metrics (Open)**: Morphological Opening (3x3px) before skeletonization
4. **Branch removed: 10 px closed 5 dilation and 1 erosion**: Morphological Closing (10px kernel, 5 dilations, 1 erosion) before skeletonization


In [3]:
# ==============================================================================
# 0. Environment Setup & Configuration
# ==============================================================================
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import datetime
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from tqdm.auto import tqdm
from IPython.display import display, HTML

# Core crack_seg imports
from crack_seg import config
from crack_seg.data_handlers.dataset_loaders import get_train_val_test_datasets
from crack_seg.data_handlers.transforms import val_transform, train_transform
from crack_seg.utils.helpers import load_model, predict_yolo_mask
from crack_seg.utils.metrics import (
    iou_score,
    dice_coefficient,
    pixel_accuracy,
    precision_score,
    recall_score,
    specificity_score,
    _skeletonize,
    crack_orientation_mae,
)

# ------------------------------------------------------------------------------
# Evaluation Parameters
# ------------------------------------------------------------------------------
SKELETON_METHOD = getattr(config, "SKELETON_METHOD", "lee")
TOLERANCE_PX = float(getattr(config, "CENTERLINE_TOLERANCE_PX", 20.0))
TOL_TAG = int(TOLERANCE_PX) if TOLERANCE_PX.is_integer() else f"{TOLERANCE_PX:.1f}"
OPEN_KERNEL_SIZE = 10
CLOSE_KERNEL_SIZE = 10
CLOSE_DILATE_ITERS = 5
CLOSE_ERODE_ITERS = 1

# Quick check option: set to an integer (e.g. 50) or None to evaluate all test samples
MAX_SAMPLES = None

print("=" * 70)
print("Crack Segmentation Evaluation Setup")
print("=" * 70)
print("PyTorch version    :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("Device             :", config.DEVICE)
print(f"Skeleton Algorithm : {SKELETON_METHOD}")
print(f"Centerline Buffer  : +/-{TOLERANCE_PX} px")
print(f"Morph Open Kernel  : {OPEN_KERNEL_SIZE}x{OPEN_KERNEL_SIZE} px")
print(f"Morph Close Kernel : {CLOSE_KERNEL_SIZE}x{CLOSE_KERNEL_SIZE} px (dilate: {CLOSE_DILATE_ITERS}, erode: {CLOSE_ERODE_ITERS})")


Crack Segmentation Evaluation Setup
PyTorch version    : 2.10.0+cu128
CUDA available     : True
Device             : cuda
Skeleton Algorithm : lee
Centerline Buffer  : +/-20.0 px
Morph Open Kernel  : 10x10 px
Morph Close Kernel : 10x10 px (dilate: 5, erode: 1)


In [4]:
# ==============================================================================
# 1. Build Test Split
# ==============================================================================
if 'test_ds' not in globals():
    print("Building dataset splits...")
    train_ds, _, test_ds = get_train_val_test_datasets(
        train_transform=train_transform,
        val_transform=val_transform,
        dataset_names=config.DATASETS,
        data_root=config.DATA_ROOT,
        train_ratio=config.TRAIN_RATIO,
        val_ratio=config.VAL_RATIO,
        test_ratio=config.TEST_RATIO,
        seed=config.SPLIT_SEED,
        stratified=config.STRATIFIED_SPLIT,
    )

    # Sanity-check: ensure no test images leaked into training
    train_paths = {img.resolve() for img, _ in train_ds.samples}
    test_paths  = {img.resolve() for img, _ in test_ds.samples}
    overlap = train_paths & test_paths
    if overlap:
        raise RuntimeError(
            f"Evaluation aborted: {len(overlap)} test image(s) also appear in training data."
        )

num_samples_to_eval = len(test_ds)
if MAX_SAMPLES is not None and MAX_SAMPLES > 0:
    num_samples_to_eval = min(MAX_SAMPLES, len(test_ds))

print(f"Total Test Set     : {len(test_ds):,} unseen images (no training overlap confirmed)")
if num_samples_to_eval != len(test_ds):
    print(f"Evaluating Subset  : {num_samples_to_eval:,} samples (MAX_SAMPLES specified)")
else:
    print(f"Evaluating All     : {num_samples_to_eval:,} samples")


Building dataset splits...

Dataset Splits Created (Total: 16089 samples):
  - Train:      11763 samples
  - Validation: 1600 samples
  - Test:       2726 samples

Total Test Set     : 2,726 unseen images (no training overlap confirmed)
Evaluating All     : 2,726 samples


In [5]:
# ==============================================================================
# 2. Model Checkpoint Paths & Loading
# ==============================================================================
MODELS_TO_EVALUATE = {
    'UNet (NCCD-PF)': {
        'type': 'unet',
        'path': config.CHECKPOINT_DIR / 'unet_NCCD-PF_Dataset.pth',
    },
    'UNet (4 Datasets)': {
        'type': 'unet',
        'path': config.CHECKPOINT_DIR / 'unet_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pth',
    },
    'SegFormer (4 Datasets)': {
        'type': 'segformer',
        'path': config.CHECKPOINT_DIR / 'segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pth',
    },
    'SegFormer (Focal)': {
        'type': 'segformer',
        'path': config.CHECKPOINT_DIR / 'segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500_focal.pth',
    },
    'SegFormer (Dice+Focal)': {
        'type': 'segformer',
        'path': config.CHECKPOINT_DIR / 'segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500_dice_focal.pth',
    },
    'YOLO-seg (4 Datasets)': {
        'type': 'yolo',
        'path': config.CHECKPOINT_DIR / 'yolo_runs' / 'yolo_seg_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pt',
    },
}

active_models = {}
print("Discovering model checkpoints:")
for model_name, info in MODELS_TO_EVALUATE.items():
    ckpt_path = Path(info['path'])
    if ckpt_path.exists():
        active_models[model_name] = info
        print(f"  [+] Found  : {model_name:<26} ({ckpt_path.name})")
    else:
        print(f"  [-] Missing: {model_name:<26} ({ckpt_path.name}) - skipping")

if not active_models:
    raise FileNotFoundError("No valid model checkpoints found. Check CHECKPOINT_DIR in config.")

# Load active models into memory
loaded_models = {}
print("\nLoading active models into memory:")
for model_name, info in active_models.items():
    print(f"  Loading {model_name}...")
    loaded_models[model_name] = (
        info['type'],
        load_model(info['type'], info['path'], device=config.DEVICE)
    )

print(f"\n[+] Successfully loaded {len(loaded_models)} model(s).")


Discovering model checkpoints:
  [+] Found  : UNet (NCCD-PF)             (unet_NCCD-PF_Dataset.pth)
  [+] Found  : UNet (4 Datasets)          (unet_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pth)
  [+] Found  : SegFormer (4 Datasets)     (segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pth)
  [+] Found  : SegFormer (Focal)          (segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500_focal.pth)
  [+] Found  : SegFormer (Dice+Focal)     (segformer_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500_dice_focal.pth)
  [+] Found  : YOLO-seg (4 Datasets)      (yolo_seg_CConCrack_NCCD-PF_Dataset_DeepCrack_CRACK500.pt)

Loading active models into memory:
  Loading UNet (NCCD-PF)...
  Loading UNet (4 Datasets)...
  Loading SegFormer (4 Datasets)...
  Loading SegFormer (Focal)...
  Loading SegFormer (Dice+Focal)...
  Loading YOLO-seg (4 Datasets)...

[+] Successfully loaded 6 model(s).


In [6]:
# ==============================================================================
# 3. Metric Keys Definition & Evaluation Loop
# ==============================================================================
area_metrics = [
    'IoU',
    'Dice',
    'Accuracy',
    'Precision',
    'Recall',
    'Specificity',
]

topo_raw_metrics = [
    'clDice',
    f'Centerline_Prec@{TOL_TAG}',
    f'Centerline_Rec@{TOL_TAG}',
    f'Centerline_F1@{TOL_TAG}',
    f'Centerline_IoU@{TOL_TAG}',
    'Orientation_MAE(deg)',
]

topo_opened_metrics = [
    'clDice (Opened)',
    f'Centerline_Prec@{TOL_TAG} (Opened)',
    f'Centerline_Rec@{TOL_TAG} (Opened)',
    f'Centerline_F1@{TOL_TAG} (Opened)',
    f'Centerline_IoU@{TOL_TAG} (Opened)',
]

topo_closed_metrics = [
    'clDice (Closed D5:E1)',
    f'Centerline_Prec@{TOL_TAG} (Closed D5:E1)',
    f'Centerline_Rec@{TOL_TAG} (Closed D5:E1)',
    f'Centerline_F1@{TOL_TAG} (Closed D5:E1)',
    f'Centerline_IoU@{TOL_TAG} (Closed D5:E1)',
]

all_metric_keys = area_metrics + topo_raw_metrics + topo_opened_metrics + topo_closed_metrics

all_model_metrics = {
    model_name: {m_name: [] for m_name in all_metric_keys}
    for model_name in loaded_models
}

print(f"Running evaluation on {num_samples_to_eval:,} test samples across {len(loaded_models)} models...")

with torch.no_grad():
    for sample_index in tqdm(range(num_samples_to_eval), desc="Evaluation Progress"):
        image_tensor, target = test_ds[sample_index]
        image_path, _ = test_ds.samples[sample_index]

        # Ground Truth processing (done ONCE per sample for efficiency)
        gt_2d = target.squeeze().cpu().numpy()
        gt_bin = (gt_2d >= 0.5).astype(np.uint8)

        # 1. Ground Truth Raw Skeleton
        gt_skel_raw = _skeletonize(
            gt_bin, method=SKELETON_METHOD, morph_open=False, prune_spurs=False
        )
        len_gt_raw = int(gt_skel_raw.sum())
        if len_gt_raw > 0:
            dist_gt_raw = cv2.distanceTransform(1 - gt_skel_raw, cv2.DIST_L2, 5)
            bound_gt_raw = (dist_gt_raw <= TOLERANCE_PX).astype(np.uint8)
        else:
            dist_gt_raw = None
            bound_gt_raw = np.zeros_like(gt_bin)

        # 2. Ground Truth Opened Skeleton (Branch Removed)
        gt_skel_open = _skeletonize(
            gt_bin,
            method=SKELETON_METHOD,
            morph_open=True,
            open_kernel_size=OPEN_KERNEL_SIZE,
            prune_spurs=False,
        )
        len_gt_open = int(gt_skel_open.sum())
        if len_gt_open > 0:
            dist_gt_open = cv2.distanceTransform(1 - gt_skel_open, cv2.DIST_L2, 5)
            bound_gt_open = (dist_gt_open <= TOLERANCE_PX).astype(np.uint8)
        else:
            dist_gt_open = None
            bound_gt_open = np.zeros_like(gt_bin)

        # 3. Ground Truth Closed Skeleton (Branch Removed: 10px, 5 dilation, 1 erosion)
        gt_skel_close = _skeletonize(
            gt_bin,
            method=SKELETON_METHOD,
            morph_op="close",
            morph_kernel_size=CLOSE_KERNEL_SIZE,
            dilate_iterations=CLOSE_DILATE_ITERS,
            erode_iterations=CLOSE_ERODE_ITERS,
            prune_spurs=False,
        )
        len_gt_close = int(gt_skel_close.sum())
        if len_gt_close > 0:
            dist_gt_close = cv2.distanceTransform(1 - gt_skel_close, cv2.DIST_L2, 5)
            bound_gt_close = (dist_gt_close <= TOLERANCE_PX).astype(np.uint8)
        else:
            dist_gt_close = None
            bound_gt_close = np.zeros_like(gt_bin)

        # Evaluate each active model
        for model_name, (m_type, model_obj) in loaded_models.items():

            # --- A. Predict Probability Map ---
            if m_type in ('unet', 'segformer'):
                output = model_obj(image_tensor.unsqueeze(0).to(config.DEVICE))
                prediction = torch.sigmoid(output).squeeze(0).cpu()   # (1, H, W)
            elif m_type == 'yolo':
                yolo_image = Image.open(image_path).convert('RGB')
                yolo_result = model_obj.predict(
                    source=yolo_image,
                    conf=0.25,
                    imgsz=config.IMG_SIZE[0],
                    verbose=False,
                )[0]
                # Merge all instance masks into single probability map
                yolo_mask = np.zeros(
                    (yolo_image.height, yolo_image.width), dtype=np.float32
                )
                if yolo_result.masks is not None:
                    for inst_mask in yolo_result.masks.data.cpu().numpy():
                        inst_img = Image.fromarray(
                            (inst_mask > 0.5).astype(np.uint8) * 255
                        ).resize(
                            (yolo_image.width, yolo_image.height),
                            Image.Resampling.NEAREST,
                        )
                        yolo_mask = np.maximum(
                            yolo_mask, np.asarray(inst_img) / 255.0
                        )
                # Resize to match dataset target shape
                th, tw = target.shape[-2:]
                yolo_pred_img = Image.fromarray(
                    (yolo_mask * 255).astype(np.uint8)
                ).resize((tw, th), Image.Resampling.NEAREST)
                prediction = (
                    torch.from_numpy(np.asarray(yolo_pred_img) / 255.0)
                    .unsqueeze(0)
                    .float()
                )  # (1, H, W)

            pred_2d = prediction.squeeze().cpu().numpy()
            pred_bin = (pred_2d >= 0.5).astype(np.uint8)

            # --- B. Area-based Metrics ---
            all_model_metrics[model_name]['IoU'].append(
                iou_score(prediction, target).item()
            )
            all_model_metrics[model_name]['Dice'].append(
                dice_coefficient(prediction, target).item()
            )
            all_model_metrics[model_name]['Accuracy'].append(
                pixel_accuracy(prediction, target).item()
            )
            all_model_metrics[model_name]['Precision'].append(
                precision_score(prediction, target).item()
            )
            all_model_metrics[model_name]['Recall'].append(
                recall_score(prediction, target).item()
            )
            all_model_metrics[model_name]['Specificity'].append(
                specificity_score(prediction, target).item()
            )

            # --- C. Orientation MAE ---
            all_model_metrics[model_name]['Orientation_MAE(deg)'].append(
                crack_orientation_mae(pred_bin, gt_bin)
            )

            # --- D. Raw Skeleton Topology & Centerline Metrics ---
            pred_skel_raw = _skeletonize(
                pred_bin, method=SKELETON_METHOD, morph_open=False, prune_spurs=False
            )
            len_pred_raw = int(pred_skel_raw.sum())

            # clDice (Raw)
            if len_pred_raw == 0 and len_gt_raw == 0:
                cldice_raw = 1.0
            elif len_pred_raw == 0 or len_gt_raw == 0:
                cldice_raw = 0.0
            else:
                tprec = (pred_skel_raw * gt_bin).sum() / (len_pred_raw + 1e-6)
                tsens = (gt_skel_raw * pred_bin).sum() / (len_gt_raw + 1e-6)
                cldice_raw = float((2.0 * tprec * tsens) / (tprec + tsens)) if (tprec + tsens) > 0 else 0.0
            all_model_metrics[model_name]['clDice'].append(cldice_raw)

            # Centerline Buffer Metrics (Raw)
            if len_pred_raw == 0 and len_gt_raw == 0:
                prec_r, rec_r, f1_r, iou_r = 1.0, 1.0, 1.0, 1.0
            elif len_pred_raw == 0 or len_gt_raw == 0:
                prec_r, rec_r, f1_r, iou_r = 0.0, 0.0, 0.0, 0.0
            else:
                dist_p_raw = cv2.distanceTransform(1 - pred_skel_raw, cv2.DIST_L2, 5)
                bound_p_raw = (dist_p_raw <= TOLERANCE_PX).astype(np.uint8)
                len_p_in_raw = int(np.logical_and(pred_skel_raw, bound_gt_raw).sum())
                len_g_cov_raw = int(np.logical_and(gt_skel_raw, bound_p_raw).sum())
                prec_r = len_p_in_raw / len_pred_raw
                rec_r = len_g_cov_raw / len_gt_raw
                f1_r = (2.0 * prec_r * rec_r) / (prec_r + rec_r) if (prec_r + rec_r) > 0 else 0.0
                inter_r = min(len_p_in_raw, len_g_cov_raw)
                iou_r = inter_r / max(len_pred_raw + len_gt_raw - inter_r, 1)

            all_model_metrics[model_name][f'Centerline_Prec@{TOL_TAG}'].append(prec_r)
            all_model_metrics[model_name][f'Centerline_Rec@{TOL_TAG}'].append(rec_r)
            all_model_metrics[model_name][f'Centerline_F1@{TOL_TAG}'].append(f1_r)
            all_model_metrics[model_name][f'Centerline_IoU@{TOL_TAG}'].append(iou_r)

            # --- E. Topology & Centerline Metrics (Opened: Branch Removed) ---
            pred_skel_open = _skeletonize(
                pred_bin,
                method=SKELETON_METHOD,
                morph_open=True,
                open_kernel_size=OPEN_KERNEL_SIZE,
                prune_spurs=False,
            )
            len_pred_open = int(pred_skel_open.sum())

            # clDice (Opened)
            if len_pred_open == 0 and len_gt_open == 0:
                cldice_open = 1.0
            elif len_pred_open == 0 or len_gt_open == 0:
                cldice_open = 0.0
            else:
                tprec_o = (pred_skel_open * gt_bin).sum() / (len_pred_open + 1e-6)
                tsens_o = (gt_skel_open * pred_bin).sum() / (len_gt_open + 1e-6)
                cldice_open = float((2.0 * tprec_o * tsens_o) / (tprec_o + tsens_o)) if (tprec_o + tsens_o) > 0 else 0.0
            all_model_metrics[model_name]['clDice (Opened)'].append(cldice_open)

            # Centerline Buffer Metrics (Opened)
            if len_pred_open == 0 and len_gt_open == 0:
                prec_o, rec_o, f1_o, iou_o = 1.0, 1.0, 1.0, 1.0
            elif len_pred_open == 0 or len_gt_open == 0:
                prec_o, rec_o, f1_o, iou_o = 0.0, 0.0, 0.0, 0.0
            else:
                dist_p_open = cv2.distanceTransform(1 - pred_skel_open, cv2.DIST_L2, 5)
                bound_p_open = (dist_p_open <= TOLERANCE_PX).astype(np.uint8)
                len_p_in_open = int(np.logical_and(pred_skel_open, bound_gt_open).sum())
                len_g_cov_open = int(np.logical_and(gt_skel_open, bound_p_open).sum())
                prec_o = len_p_in_open / len_pred_open
                rec_o = len_g_cov_open / len_gt_open
                f1_o = (2.0 * prec_o * rec_o) / (prec_o + rec_o) if (prec_o + rec_o) > 0 else 0.0
                inter_o = min(len_p_in_open, len_g_cov_open)
                iou_o = inter_o / max(len_pred_open + len_gt_open - inter_o, 1)

            all_model_metrics[model_name][f'Centerline_Prec@{TOL_TAG} (Opened)'].append(prec_o)
            all_model_metrics[model_name][f'Centerline_Rec@{TOL_TAG} (Opened)'].append(rec_o)
            all_model_metrics[model_name][f'Centerline_F1@{TOL_TAG} (Opened)'].append(f1_o)
            all_model_metrics[model_name][f'Centerline_IoU@{TOL_TAG} (Opened)'].append(iou_o)

            # --- F. Topology & Centerline Metrics (Branch Removed: 10px Closed D5:E1) ---
            pred_skel_close = _skeletonize(
                pred_bin,
                method=SKELETON_METHOD,
                morph_op="close",
                morph_kernel_size=CLOSE_KERNEL_SIZE,
                dilate_iterations=CLOSE_DILATE_ITERS,
                erode_iterations=CLOSE_ERODE_ITERS,
                prune_spurs=False,
            )
            len_pred_close = int(pred_skel_close.sum())

            # clDice (Closed)
            if len_pred_close == 0 and len_gt_close == 0:
                cldice_close = 1.0
            elif len_pred_close == 0 or len_gt_close == 0:
                cldice_close = 0.0
            else:
                tprec_c = (pred_skel_close * gt_bin).sum() / (len_pred_close + 1e-6)
                tsens_c = (gt_skel_close * pred_bin).sum() / (len_gt_close + 1e-6)
                cldice_close = float((2.0 * tprec_c * tsens_c) / (tprec_c + tsens_c)) if (tprec_c + tsens_c) > 0 else 0.0
            all_model_metrics[model_name]['clDice (Closed D5:E1)'].append(cldice_close)

            # Centerline Buffer Metrics (Closed)
            if len_pred_close == 0 and len_gt_close == 0:
                prec_c, rec_c, f1_c, iou_c = 1.0, 1.0, 1.0, 1.0
            elif len_pred_close == 0 or len_gt_close == 0:
                prec_c, rec_c, f1_c, iou_c = 0.0, 0.0, 0.0, 0.0
            else:
                dist_p_close = cv2.distanceTransform(1 - pred_skel_close, cv2.DIST_L2, 5)
                bound_p_close = (dist_p_close <= TOLERANCE_PX).astype(np.uint8)
                len_p_in_close = int(np.logical_and(pred_skel_close, bound_gt_close).sum())
                len_g_cov_close = int(np.logical_and(gt_skel_close, bound_p_close).sum())
                prec_c = len_p_in_close / len_pred_close
                rec_c = len_g_cov_close / len_gt_close
                f1_c = (2.0 * prec_c * rec_c) / (prec_c + rec_c) if (prec_c + rec_c) > 0 else 0.0
                inter_c = min(len_p_in_close, len_g_cov_close)
                iou_c = inter_c / max(len_pred_close + len_gt_close - inter_c, 1)

            all_model_metrics[model_name][f'Centerline_Prec@{TOL_TAG} (Closed D5:E1)'].append(prec_c)
            all_model_metrics[model_name][f'Centerline_Rec@{TOL_TAG} (Closed D5:E1)'].append(rec_c)
            all_model_metrics[model_name][f'Centerline_F1@{TOL_TAG} (Closed D5:E1)'].append(f1_c)
            all_model_metrics[model_name][f'Centerline_IoU@{TOL_TAG} (Closed D5:E1)'].append(iou_c)

print(f"\n[+] Completed evaluation of {num_samples_to_eval:,} samples.")


Running evaluation on 2,726 test samples across 6 models...


Evaluation Progress: 100%|██████████| 2726/2726 [22:42<00:00,  2.00it/s]


[+] Completed evaluation of 2,726 samples.


In [7]:
# ==============================================================================
# 4. Format & Display Comparison Table
# ==============================================================================
mean_metrics = {
    model_name: {
        metric_name: float(np.mean(values))
        for metric_name, values in metrics.items()
    }
    for model_name, metrics in all_model_metrics.items()
}

COL_W  = max(24, max(len(name) + 2 for name in loaded_models))   # dynamic width per model column
NAME_W = 36   # width for metric name column

header = f"{'Metric':<{NAME_W}}" + "".join(
    f"{name:>{COL_W}}" for name in loaded_models
)
sep  = "=" * len(header)
dash = "-" * len(header)

print(f"\nEvaluation Results - {num_samples_to_eval:,} Unseen Test Samples")
print(sep)
print(header)

# Section 1: Area-based
print(dash)
print(f"{'--- 1. Area-based ---':<{NAME_W}}")
print(dash)
for m_name in area_metrics:
    row = f"{m_name:<{NAME_W}}"
    for model_name in loaded_models:
        row += f"{mean_metrics[model_name][m_name]:>{COL_W}.4f}"
    print(row)

# Section 2: Topology & Centerline (Raw Skeleton)
print(dash)
print(f"{'--- 2. Topology & Centerline (Raw Skeleton) ---':<{NAME_W}}")
print(dash)
for m_name in topo_raw_metrics:
    row = f"{m_name:<{NAME_W}}"
    for model_name in loaded_models:
        row += f"{mean_metrics[model_name][m_name]:>{COL_W}.4f}"
    print(row)

# Section 3: Topology & Centerline (Branch Removed via Morphological Opening)
print(dash)
print(f"{f'--- 3. Topology & Centerline (Branch Removed: Open {OPEN_KERNEL_SIZE}px) ---':<{NAME_W}}")
print(dash)
for m_name in topo_opened_metrics:
    row = f"{m_name:<{NAME_W}}"
    for model_name in loaded_models:
        row += f"{mean_metrics[model_name][m_name]:>{COL_W}.4f}"
    print(row)

# Section 4: Topology & Centerline (Branch removed: 10 px closed 5 dilation and 1 erosion)
print(dash)
print(f"{'--- 4. Branch removed: 10 px closed 5 dilation and 1 erosion ---':<{NAME_W}}")
print(dash)
for m_name in topo_closed_metrics:
    row = f"{m_name:<{NAME_W}}"
    for model_name in loaded_models:
        row += f"{mean_metrics[model_name][m_name]:>{COL_W}.4f}"
    print(row)

print(sep)
print("Metric Notes & Explanations:")
print(f"  * Raw vs. Branch-Removed    : Section 2 uses raw skeletons. Section 3 uses {OPEN_KERNEL_SIZE}x{OPEN_KERNEL_SIZE} opening.")
print(f"  * Section 4 (Closed D5:E1)   : Uses {CLOSE_KERNEL_SIZE}px closing with {CLOSE_DILATE_ITERS} dilations and {CLOSE_ERODE_ITERS} erosion before skeletonization.")
print(f"  * Centerline_*@{TOL_TAG}           : Width-independent precision/recall/F1/IoU within +/-{TOLERANCE_PX}px buffer.")
print(f"  * clDice                     : Centerline Dice (CVPR 2021) measuring topological connectivity.")
print(f"  * Orientation_MAE            : Mean Angular Error in degrees (lower is better, 0 deg = perfect alignment).\n")

# Render as styled HTML Table in Notebook
model_names = list(loaded_models.keys())
html = [
    "<div style='margin: 15px 0;'>",
    "<table style='border-collapse: collapse; font-family: -apple-system, BlinkMacSystemFont, Arial, sans-serif; width: 100%; border: 1px solid #d0d7de;'>",
    "<thead><tr style='background-color: #24292f; color: white;'>",
    "<th style='padding: 10px 14px; text-align: left; font-size: 14px;'>Metric</th>"
]
for m in model_names:
    html.append(f"<th style='padding: 10px 14px; text-align: right; font-size: 14px;'>{m}</th>")
html.append("</tr></thead><tbody>")

def add_html_section(title, metrics_list, bg_color):
    html.append(f"<tr style='background-color: {bg_color}; font-weight: 600; color: #24292f;'>")
    html.append(f"<td colspan='{1 + len(model_names)}' style='padding: 9px 14px; font-size: 13px;'>{title}</td></tr>")
    for m in metrics_list:
        html.append("<tr style='border-bottom: 1px solid #e1e4e8;'>")
        html.append(f"<td style='padding: 8px 14px; text-align: left; font-size: 13px;'>{m}</td>")
        for model in model_names:
            val = mean_metrics[model][m]
            html.append(f"<td style='padding: 8px 14px; text-align: right; font-family: SFMono-Regular, Consolas, monospace; font-size: 13px;'>{val:.4f}</td>")
        html.append("</tr>")

add_html_section("1. Area-based Metrics", area_metrics, "#ddf4ff")
add_html_section("2. Topology & Centerline Metrics (Raw Skeleton)", topo_raw_metrics, "#fff8c5")
add_html_section(f"3. Topology & Centerline Metrics (Branch Removed: {OPEN_KERNEL_SIZE}px Open)", topo_opened_metrics, "#dafbe1")
add_html_section("4. Branch removed: 10 px closed 5 dilation and 1 erosion", topo_closed_metrics, "#fbe8ff")

html.append("</tbody></table></div>")
display(HTML("".join(html)))



Evaluation Results - 2,726 Unseen Test Samples
Metric                                        UNet (NCCD-PF)       UNet (4 Datasets)  SegFormer (4 Datasets)       SegFormer (Focal)  SegFormer (Dice+Focal)   YOLO-seg (4 Datasets)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
--- 1. Area-based ---               
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
IoU                                                   0.1513                  0.5587                  0.6383                  0.4927                  0.6682                  0.4728
Dice                                                  0.2138                  0.6906                  0.7646                  0.6093                  0.7894                  0.5856
Accuracy  